# Train Squad's own model on a free Colab GPU

Fine-tunes Qwen3.5-0.8B with LoRA for all of Peek's jobs (routing, card titles, recaps, asking back), scores it on every test set, and gives you one ~0.5 GB `peek-q4_k_m.gguf` file to download.

**First:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all. With all the teacher data it takes about 1.5 to 2 hours; keep the tab open. If a step stops, run it again: training carries on from its last checkpoint.

No API key is needed here: the teacher's examples are already in `training/data/`.

In [ ]:
# 1. The GPU and the code
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, getpass
REPO = "github.com/cosmospan/squadcode"
BRANCH = "claude/project-thread-8o4ttl"
os.chdir("/content")
if not os.path.isdir("squadcode"):
    r = os.system(f"git clone -q --depth 1 -b {BRANCH} https://{REPO} squadcode")
    if r != 0:  # a private repo needs a GitHub token with read access
        token = getpass.getpass("GitHub token (read access to the repo): ")
        os.system(f"git clone -q --depth 1 -b {BRANCH} https://x-access-token:{token}@{REPO} squadcode")
%cd /content/squadcode/training

In [ ]:
# 2. Packages. Colab's own torchao is too old for the newest PEFT.
!pip install -q -U transformers peft trl datasets accelerate "torchao>=0.16.0"

In [ ]:
# 3. Training examples from the seeds (no API key)
!python build_data.py --out work/data/synth.jsonl

In [ ]:
# 4. Train (about 1 to 1.5 hours on a T4 with all the data). Every file in
# data/ joins the seeds: routing, titles, recaps and asking back.
import glob
DATA = " ".join(["work/data/synth.jsonl"] + sorted(glob.glob("data/*.jsonl")))
print("Training on:", DATA)
!python train_lora.py --data {DATA} --out work/peek-lora


In [ ]:
# 5. Score it on every test set, as the app asks it
import os
os.chdir("/content/squadcode/training")
!python score.py --hf work/peek-lora/merged
# Peek v1 got route 53/53, route_hard 120/120. Ship only if v2 matches those
# and does better on route_wild.


In [ ]:
# 6. Shrink to one GGUF file (about 0.5 GB)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp work/llama.cpp
# Not llama.cpp's requirements file: it pins an older transformers that
# can't read Qwen3.5's tokenizer. The converter brings its own gguf module.
!pip install -q sentencepiece
!python work/llama.cpp/convert_hf_to_gguf.py work/peek-lora/merged --outfile work/peek-f16.gguf --outtype f16
!cmake -S work/llama.cpp -B work/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build work/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!work/llama.cpp/build/bin/llama-quantize work/peek-f16.gguf work/peek-q4_k_m.gguf Q4_K_M
!ls -lh work/peek-q4_k_m.gguf

In [ ]:
# 7. Download it
from google.colab import files
files.download("work/peek-q4_k_m.gguf")

## On your Mac

Put the file somewhere (say `~/Downloads`), then:

```
brew install llama.cpp
llama-server -m ~/Downloads/peek-q4_k_m.gguf --port 8088 --alias peek --chat-template-kwargs '{"enable_thinking":false}' -c 4096 -np 1
```

The flags matter: Qwen3.5 thinks first by default and runs out of room before it answers, and without `-c 4096 -np 1` the server reserves 3.3 GB of memory instead of about 250 MB.

In `~/.squadcode/settings.json` set `"brain_model": "peek"`, and Squad uses it. `SquadCode v2 brain eval` scores it against the rules.